# Stage 2: Dataset Integration & Phase Analysis

**Project:** KAN-Driven Phase-Spectrum Analysis for Next-Generation Deepfake Detection  
**Environment:** Kaggle Notebook (GPU T4)  
**Dataset:** Mounted at `/kaggle/input/artifact-dataset`

In [ ]:
# =============================================================================
# Cell 1: Imports & Kaggle Paths
# =============================================================================
import numpy as np
import pandas as pd
import cv2
import os
import glob
from typing import Dict, List, Tuple, Optional
from collections import defaultdict
from scipy import stats
from skimage import img_as_float
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'serif'
sns.set_style('whitegrid')

# --- Kaggle Paths ---
INPUT_DIR = '/kaggle/input/artifact-dataset'   # Read-only dataset
OUTPUT_DIR = '/kaggle/working'                  # Writable output
CACHE_DIR = os.path.join(OUTPUT_DIR, 'phase_cache')
os.makedirs(CACHE_DIR, exist_ok=True)

print('Imports OK')
print(f'Dataset : {INPUT_DIR}')
print(f'Cache   : {CACHE_DIR}')

# Directory overview
print('\nTop-level contents:')
for item in sorted(os.listdir(INPUT_DIR)):
    full = os.path.join(INPUT_DIR, item)
    kind = 'DIR ' if os.path.isdir(full) else 'FILE'
    print(f'  [{kind}] {item}')

In [ ]:
# =============================================================================
# Cell 2: Dataset Discovery & Metadata Loading
# =============================================================================


class DatasetExplorer:
    def __init__(self, dataset_root: str) -> None:
        self.dataset_root = dataset_root
        self.metadata_df = None
    
    def discover_metadata_files(self) -> List[str]:
        pattern = os.path.join(self.dataset_root, '**', 'metadata.csv')
        files = glob.glob(pattern, recursive=True)
        print(f'Found {len(files)} metadata.csv file(s)')
        return sorted(files)
    
    def load_metadata(self) -> pd.DataFrame:
        meta_files = self.discover_metadata_files()
        all_dfs = []
        for meta_path in meta_files:
            generator_dir = os.path.dirname(meta_path)
            generator_name = os.path.basename(generator_dir)
            df = pd.read_csv(meta_path)
            df['generator'] = generator_name
            df['image_path'] = df['image_path'].apply(
                lambda p: os.path.join(generator_dir, p) if not os.path.isabs(p) else p)
            all_dfs.append(df)
            print(f'  {len(df):>6d} entries from [{generator_name}]')
        
        if not all_dfs:
            return self._fallback_scan()
        
        self.metadata_df = pd.concat(all_dfs, ignore_index=True)
        if 'target' in self.metadata_df.columns:
            self.metadata_df['target'] = self.metadata_df['target'].astype(int)
        return self.metadata_df
    
    def _fallback_scan(self) -> pd.DataFrame:
        image_exts = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.webp'}
        records = []
        for root, dirs, files in os.walk(self.dataset_root):
            for f in files:
                if os.path.splitext(f)[1].lower() in image_exts:
                    full_path = os.path.join(root, f)
                    parts = os.path.relpath(root, self.dataset_root).lower().split(os.sep)
                    if 'real' in parts:
                        target, generator = 0, 'real'
                    elif 'fake' in parts:
                        target = 1
                        generator = parts[-1] if parts[-1] != 'fake' else 'unknown_fake'
                    else:
                        target, generator = -1, parts[0] if parts else 'unknown'
                    records.append({'image_path': full_path, 'target': target,
                                    'category': 'unknown', 'generator': generator})
        self.metadata_df = pd.DataFrame(records)
        print(f'  Fallback scan: {len(self.metadata_df)} images')
        return self.metadata_df
    
    def print_summary(self) -> None:
        df = self.metadata_df
        if df is None:
            return
        print('=' * 60)
        print(f'Total images: {len(df):,}')
        if 'target' in df.columns:
            print(f'Real: {(df["target"]==0).sum():,} | Fake: {(df["target"]==1).sum():,}')
        if 'generator' in df.columns:
            print(f'\nGenerators ({df["generator"].nunique()}):')
            for gen, count in df['generator'].value_counts().items():
                label = 'REAL' if len(df[df['generator']==gen]) > 0 and df[df['generator']==gen]['target'].mode().iloc[0] == 0 else 'FAKE'
                print(f'  {gen:30s}: {count:>6,} [{label}]')
        print('=' * 60)


explorer = DatasetExplorer(INPUT_DIR)
metadata_df = explorer.load_metadata()
explorer.print_summary()
metadata_df.head()

In [ ]:
# =============================================================================
# Cell 3: Batch Phase Extraction
# =============================================================================


class PhaseSpectrumExtractor:
    _LUMA_R, _LUMA_G, _LUMA_B = 0.2989, 0.5870, 0.1140
    def __init__(self, target_size=(256, 256)):
        self.target_size = target_size
    def process_single(self, image_path):
        try:
            bgr = cv2.imread(image_path, cv2.IMREAD_COLOR)
            if bgr is None: return None
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            img_f = rgb.astype(np.float64)
            gray = self._LUMA_R*img_f[:,:,0] + self._LUMA_G*img_f[:,:,1] + self._LUMA_B*img_f[:,:,2]
            gray = cv2.resize(gray, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_CUBIC)
            fft_shifted = np.fft.fftshift(np.fft.fft2(gray))
            phase = np.angle(fft_shifted)
            p_min, p_max = phase.min(), phase.max()
            if p_max - p_min == 0: return np.zeros(self.target_size, dtype=np.float64)
            return (phase - p_min) / (p_max - p_min)
        except Exception:
            return None


class BatchPhaseExtractor:
    def __init__(self, cache_dir, target_size=(256, 256)):
        self.cache_dir = cache_dir
        self.extractor = PhaseSpectrumExtractor(target_size)
        os.makedirs(cache_dir, exist_ok=True)
    
    def extract_batch(self, df, n_samples_per_generator=50, use_cache=True):
        cache_file = os.path.join(self.cache_dir, f'phase_maps_n{n_samples_per_generator}.npy')
        if use_cache and os.path.exists(cache_file):
            print(f'Loading cache: {cache_file}')
            results = np.load(cache_file, allow_pickle=True).item()
            for gen, maps in results.items():
                print(f'  {gen:30s}: {len(maps)} maps')
            return results
        
        generators = df['generator'].unique()
        results = {}
        print(f'Processing {len(generators)} generators ({n_samples_per_generator} samples each)...')
        
        for gen in sorted(generators):
            gen_df = df[df['generator'] == gen]
            if n_samples_per_generator > 0 and len(gen_df) > n_samples_per_generator:
                gen_df = gen_df.sample(n=n_samples_per_generator, random_state=42)
            phase_maps, failed = [], 0
            for img_path in tqdm(gen_df['image_path'].values, desc=f'{gen[:25]:25s}', leave=True):
                phase = self.extractor.process_single(img_path)
                if phase is not None: phase_maps.append(phase)
                else: failed += 1
            if phase_maps:
                results[gen] = np.array(phase_maps)
                print(f'  {gen}: {len(phase_maps)} OK, {failed} failed')
        
        print(f'Saving cache: {cache_file}')
        np.save(cache_file, results)
        return results


# --- Execute ---
N_SAMPLES = 50  # Adjust: 50=fast, 200=moderate, -1=full

batch_extractor = BatchPhaseExtractor(cache_dir=CACHE_DIR)
phase_results = batch_extractor.extract_batch(metadata_df, n_samples_per_generator=N_SAMPLES)

print('\n-- Summary --')
for gen, maps in sorted(phase_results.items()):
    print(f'  {gen:30s}: {maps.shape}')

In [ ]:
# =============================================================================
# Cell 4: Per-Generator Mean Phase Maps & Visualization
# =============================================================================

# Identify real vs fake generators
real_gens, fake_gens = [], []
for gen in phase_results:
    gen_df = metadata_df[metadata_df['generator'] == gen]
    if len(gen_df) > 0 and gen_df['target'].mode().iloc[0] == 0:
        real_gens.append(gen)
    else:
        fake_gens.append(gen)

if not real_gens:
    real_gens = [g for g in phase_results if 'real' in g.lower()]
    fake_gens = [g for g in phase_results if g not in real_gens]

# Mean maps
mean_maps = {gen: np.mean(maps, axis=0) for gen, maps in phase_results.items()}
real_all = np.concatenate([phase_results[g] for g in real_gens if g in phase_results])
real_mean = np.mean(real_all, axis=0) if len(real_all) > 0 else None

print(f'Real generators: {real_gens}')
print(f'Fake generators: {fake_gens}')

# Visualize
display_gens = fake_gens[:8]
n_cols = min(len(display_gens) + 1, 9)
n_rows = 2 if real_mean is not None else 1

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows))
if n_rows == 1: axes = axes[np.newaxis, :]
if n_cols == 1: axes = axes[:, np.newaxis]

fig.suptitle('Per-Generator Mean Phase Maps', fontsize=14, fontweight='bold', y=1.02)

if real_mean is not None:
    axes[0,0].imshow(real_mean, cmap='twilight', vmin=0, vmax=1)
    axes[0,0].set_title('REAL', fontsize=9, fontweight='bold', color='green')
    axes[0,0].axis('off')

for i, gen in enumerate(display_gens):
    col = i + 1
    if col >= n_cols: break
    axes[0, col].imshow(mean_maps[gen], cmap='twilight', vmin=0, vmax=1)
    axes[0, col].set_title(gen[:15], fontsize=8)
    axes[0, col].axis('off')

if n_rows == 2 and real_mean is not None:
    axes[1,0].text(0.5, 0.5, 'REFERENCE', ha='center', va='center',
                   fontsize=10, transform=axes[1,0].transAxes)
    axes[1,0].axis('off')
    for i, gen in enumerate(display_gens):
        col = i + 1
        if col >= n_cols: break
        diff = np.abs(mean_maps[gen] - real_mean)
        im = axes[1, col].imshow(diff, cmap='hot', vmin=0)
        axes[1, col].set_title(f'|{gen[:12]}-real|', fontsize=8)
        axes[1, col].axis('off')
    fig.colorbar(im, ax=axes[1,:].tolist(), fraction=0.02, pad=0.04, label='|Diff|')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Cell 5: KS Statistical Tests
# =============================================================================

# Build real distribution
real_samples = []
for gen in real_gens:
    if gen in phase_results:
        for m in phase_results[gen]:
            flat = m.flatten()
            idx = np.random.RandomState(42).choice(len(flat), size=min(5000, len(flat)), replace=False)
            real_samples.append(flat[idx])
real_dist = np.concatenate(real_samples)
print(f'Real distribution: {len(real_dist):,} samples')

# KS tests
ks_records = []
for gen in sorted(fake_gens):
    if gen not in phase_results: continue
    fake_samples = []
    for m in phase_results[gen]:
        flat = m.flatten()
        idx = np.random.RandomState(42).choice(len(flat), size=min(5000, len(flat)), replace=False)
        fake_samples.append(flat[idx])
    fake_dist = np.concatenate(fake_samples)
    ks_stat, p_value = stats.ks_2samp(real_dist, fake_dist)
    ks_records.append({'generator': gen, 'n_images': len(phase_results[gen]),
                       'ks_statistic': round(ks_stat, 6), 'p_value': p_value,
                       'significant': 'YES' if p_value < 0.05 else 'NO'})

ks_df = pd.DataFrame(ks_records).sort_values('ks_statistic', ascending=False).reset_index(drop=True)
print('\n-- KS Test Results --')
print(ks_df.to_string(index=False))

n_sig = (ks_df['significant'] == 'YES').sum()
print(f'\n{n_sig}/{len(ks_df)} generators show significant phase differences (p < 0.05)')
if n_sig > 0:
    print('CONCLUSION: Phase-based detection IS viable.')

# Bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(4, len(ks_df)*0.5)))
fig.suptitle('KS Test: Real vs Fake Phase Distributions', fontsize=13, fontweight='bold')
colors = ['#e74c3c' if s=='YES' else '#95a5a6' for s in ks_df['significant']]
ax1.barh(ks_df['generator'], ks_df['ks_statistic'], color=colors)
ax1.set_xlabel('KS Statistic'); ax1.set_title('KS Statistic'); ax1.invert_yaxis()
p_vals = ks_df['p_value'].replace(0, 1e-300)
ax2.barh(ks_df['generator'], -np.log10(p_vals), color=colors)
ax2.axvline(x=-np.log10(0.05), color='black', linestyle='--', label='p=0.05')
ax2.set_xlabel('-log10(p-value)'); ax2.set_title('Significance'); ax2.legend(); ax2.invert_yaxis()
plt.tight_layout()
plt.show()